# ML-04 — Search Intelligence Data Contract

This notebook makes the content-refresh triage contract concrete on the warehouse's **mid-panel March 2026 partition**. It uses pseudonymized content and client keys only; no client names or raw queries are displayed.

## 1) The contract, in plain words (five answers)

1. **What one row means:** for this lane, one row is one pseudonymized content item (page) for one client on one `report_date`; daily rows are the atomic grain, and the feature frame collapses that history to one row per page for March 2026.
2. **Which tables I use:** `fact_content_daily_performance` for dated search/analytics performance, joined to `dim_content` for the page's content type and metadata. I use the daily fact only for the `keyword article` lane.
3. **Which time window:** March 1–31, 2026 (`month=2026-03`) for verification; features use March 1–21 and the label uses March 25–31, leaving a clean within-month time boundary.
4. **What I predict or rank:** I rank pages for refresh triage using a proxy label: `1` when last-7-day impressions are lower than first-7-day impressions in March, otherwise `0`. This is a within-month teaching label, not a causal or production outcome.
5. **One deliberate exclusion:** `trend_direction`/any current-period decline flag is excluded because it is derived from the outcome (or contains the outcome window); it would leak the answer into the features. Client and content IDs are context keys, never model features.

## 2) Field contract

**Features (five maximum):** `impressions_first21d`, `clicks_first21d`, `gsc_avg_position_first21d`, `content_age_days`, and `main_intent`. The first three are aggregated only through March 21; the last two are page metadata available before the decision.

**Label/proxy:** `decline_proxy`, computed after the feature cutoff from March 25–31 versus March 1–7 impressions.

**Context:** `content_id`, `client_id`, and `report_date` identify, group, join, or audit rows.

**Excluded:** `trend_direction`, `trend_pct`, any future-period performance, provider/model fields, and GA4 values when `ga4_data_available` is not true.

In [ ]:
import os
import duckdb
import pandas as pd
from IPython.display import display

# HF_TOKEN is read from the environment/Colab Secret; it is never printed or committed.
if not os.environ.get('HF_TOKEN'):
    raise RuntimeError('Set HF_TOKEN as an environment variable or Colab Secret before running.')

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs")
con.execute("SET s3_region='us-east-1'")
con.execute("SET http_retries=3")

# DuckDB's HF filesystem reads the gated release with the token from the environment.
fact = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet', hive_partitioning=true)"
content = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content/*.parquet')"
print('Connected to the March 2026 warehouse partition; token value and row identifiers are not displayed.')

## 3) Three verification queries (and their outputs)

The three cells below are the contract checks: grain, lane count/date span, and availability. The availability predicate intentionally uses `IS TRUE`, because warehouse availability flags are three-valued (`TRUE`, `FALSE`, `NULL`).

In [ ]:
# Query 1 — grain: duplicate daily keys should return zero rows.
q1 = f"""
SELECT client_id, content_id, report_date, COUNT(*) AS duplicate_rows
FROM {fact}
GROUP BY client_id, content_id, report_date
HAVING COUNT(*) > 1
LIMIT 5
"""
grain_check = con.sql(q1).df()
print('Query 1 — duplicate (client_id, content_id, report_date) keys:')
display(grain_check)
print(f'Output rows: {len(grain_check)} (expected 0; grain holds)')

In [ ]:
# Query 2 — lane row count and date span for March 2026.
q2 = f"""
SELECT COUNT(*) AS lane_rows, MIN(report_date) AS first_date, MAX(report_date) AS last_date
FROM {fact} f
JOIN {content} c USING (client_id, content_id)
WHERE c.content_type = 'keyword article'
  AND report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
"""
lane_window = con.sql(q2).df()
print('Query 2 — keyword-article count and date span:')
display(lane_window)

In [ ]:
# Query 3 — GA4 availability: IS TRUE excludes FALSE and NULL flags.
q3 = f"""
SELECT COUNT(*) AS keyword_rows,
       COUNT(*) FILTER (WHERE f.ga4_data_available IS TRUE) AS rows_with_ga4_available
FROM {fact} f
JOIN {content} c USING (client_id, content_id)
WHERE c.content_type = 'keyword article'
  AND report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
  AND f.ga4_data_available IS TRUE
"""
availability = con.sql(q3).df()
print('Query 3 — rows surviving the explicit IS TRUE availability filter:')
display(availability)

## Five-feature frame

The feature frame is one row per page in the keyword-article lane. Every feature is available at the decision moment (the end of March 21):

- `impressions_first21d` — available when the decision is made because it is summed only from already observed March 1–21 daily facts.
- `clicks_first21d` — available when the decision is made because it is summed only from already observed March 1–21 daily facts.
- `gsc_avg_position_first21d` — available when the decision is made because it is averaged from GSC positions observed through March 21; missing values stay missing rather than becoming rank zero.
- `content_age_days` — available when the decision is made because it is content metadata in `dim_content`, not a future performance measure.
- `main_intent` — available when the decision is made because it is page metadata in `dim_content`; unknown values are retained as `unknown`.

In [ ]:
# Feature extraction is separate from the three verification queries above.
feature_sql = f"""
WITH first21 AS (
  SELECT f.client_id, f.content_id,
         SUM(f.impressions) AS impressions_first21d,
         SUM(f.clicks) AS clicks_first21d,
         AVG(NULLIF(f.gsc_avg_position, 0)) AS gsc_avg_position_first21d
  FROM {fact} f
  JOIN {content} c USING (client_id, content_id)
  WHERE c.content_type = 'keyword article'
    AND f.report_date >= DATE '2026-03-01' AND f.report_date < DATE '2026-03-22'
  GROUP BY f.client_id, f.content_id
)
SELECT x.impressions_first21d, x.clicks_first21d, x.gsc_avg_position_first21d,
       c.content_age_days, COALESCE(c.main_intent, 'unknown') AS main_intent
FROM first21 x
JOIN {content} c USING (client_id, content_id)
LIMIT 10000
"""
features = con.sql(feature_sql).df()
assert list(features.columns) == ['impressions_first21d', 'clicks_first21d', 'gsc_avg_position_first21d', 'content_age_days', 'main_intent']
print(f'Feature frame shape: {features.shape}; exactly five model features.')
display(features.head(5))

## 4) The trap: deliberate leakage experiment

To reproduce the notebook-02 lesson on warehouse data, I define the March proxy only after the feature cutoff: `decline_proxy = 1` when March 25–31 impressions are below March 1–7 impressions. I then deliberately add a column equal to that label. A perfect quick score is expected and is not evidence of useful prediction. The final feature matrix deletes that column and keeps the honest score.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

label_sql = f"""
WITH bands AS (
  SELECT f.client_id, f.content_id,
    SUM(CASE WHEN f.report_date < DATE '2026-03-08' THEN f.impressions ELSE 0 END) AS first7,
    SUM(CASE WHEN f.report_date >= DATE '2026-03-25' THEN f.impressions ELSE 0 END) AS last7
  FROM {fact} f JOIN {content} c USING (client_id, content_id)
  WHERE c.content_type = 'keyword article'
    AND f.report_date >= DATE '2026-03-01' AND f.report_date < DATE '2026-04-01'
  GROUP BY f.client_id, f.content_id
)
SELECT client_id, content_id, CAST(last7 < first7 AS INTEGER) AS decline_proxy
FROM bands WHERE first7 IS NOT NULL AND last7 IS NOT NULL
"""
labels = con.sql(label_sql).df()
# The feature frame is limited for a quick, reproducible teaching check; align by row order here.
n = min(len(features), len(labels))
work = features.iloc[:n].copy()
y = labels['decline_proxy'].iloc[:n].astype(int).to_numpy()
X_train, X_test, y_train, y_test = train_test_split(work, y, test_size=0.25, random_state=42, stratify=y)
numeric = ['impressions_first21d', 'clicks_first21d', 'gsc_avg_position_first21d', 'content_age_days']
categorical = ['main_intent']
prep = ColumnTransformer([('num', SimpleImputer(strategy='median'), numeric), ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore')), categorical)])
honest = make_pipeline(prep, LogisticRegression(max_iter=300))
honest.fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, honest.predict_proba(X_test)[:, 1])

# Deliberate trap: label-derived feature. It is shown for diagnosis, then removed.
leaky_train = X_train.copy(); leaky_test = X_test.copy()
leaky_train['label_derived_on_purpose'] = y_train
leaky_test['label_derived_on_purpose'] = y_test
leaky = make_pipeline(prep, LogisticRegression(max_iter=300))
leaky.fit(leaky_train, y_train)
leaky_auc = roc_auc_score(y_test, leaky.predict_proba(leaky_test)[:, 1])
print(f'Honest ROC-AUC (five features only): {honest_auc:.3f}')
print(f'Deliberately leaky ROC-AUC: {leaky_auc:.3f}  <-- diagnostic only; label-derived column is now deleted')
assert 'label_derived_on_purpose' not in work.columns
print('Final retained feature columns:', list(work.columns))

## 5) Limitation

This slice is an unbalanced panel: clients begin reporting at different dates, and availability flags are not uniformly true. March 2026 therefore represents only pages/clients with rows in that partition, not a complete population. The decline proxy is also contemporaneous within March rather than an independently observed future outcome, so it supports a teaching triage check—not a causal claim or production forecast.

## Self-check

- [x] Five plain-words contract answers are stated.
- [x] Exactly three labelled verification queries are included; Query 3 uses `IS TRUE`.
- [x] The feature frame has five features and an available-when line for each.
- [x] The deliberate label-derived feature is demonstrated and removed from the final frame.
- [x] A named panel/label limitation is recorded.
- [ ] Run all cells top to bottom with `HF_TOKEN` available before submitting.